# Explore and request point observations data

This notebook provides a walk-through of some example functionality for exploring and requesting point observations data and site-level metadata from the [HydroData Data Catalog](https://hydroframe.org/hydrodata) using the `hf_hydrodata` Python package. We will use hf_hydrodata's `get_site_variables`, `get_point_metadata` and `get_point_data` functions. We will also review how to properly attribute/cite the data acquired. 

The `hf_hydrodata` documentation has more complete information on which [point observation datasets](https://hf-hydrodata.readthedocs.io/en/latest/available_datasets.html#point-observations) are supported and what [site metadata features](https://hf-hydrodata.readthedocs.io/en/latest/available_metadata.html#point-observations-metadata) are available. There is also [full API documentation](https://hf-hydrodata.readthedocs.io/en/latest/api_reference.html) available for all of the `hf_hydrodata` functions. The API documentation is the most up-to-date resource for supported parameters and function options. 

### Set-Up

In [ ]:
import pandas as pd
import hf_hydrodata as hf
from matplotlib import pyplot as plt
import plotly.express as px
import plotly.io as pio


You'll need to register for a HydroFrame Account and PIN in order to use the `hf_hydrodata` package. The [Getting Started page](https://hf-hydrodata.readthedocs.io/en/latest/getting_started.html) has full details on how to sign up for an account and set up a PIN.

Once you create a PIN, you'll need to register your PIN.

In [ ]:
hf.register_api_pin("your_email", "your_api_pin")

### Example 1: Explore available observations

In this example, we will use the `get_site_variables` function to explore all of the available observations within a geographic area. This function is best used for exploration and does not require the same parameter specificity as `get_point_metadata` and `get_point_data` (more on that later). This means that you can search across variable, dataset, and temporal resolution to get a higher-level overview of how many observations are available within this region.

For these examples, we will use the `huc_id` parameter paired with `grid="conus2"` in order to filter observations to a specific Hydrologic Unit Code(HUC) area within the CONUS2 domain.

Note that there are other ways to filter observations on geography such as by specific `site_ids`, `state`, `latitude_range`, `longitude_range`, `grid_bounds`, or a user-provided `polygon` and `polygon_crs`. The [API Documentation for the point module](https://hf-hydrodata.readthedocs.io/en/latest/hf_hydrodata.point.html#module-hf_hydrodata.point) outlines all of the available options. There are also some [How-To example notebooks](https://hf-hydrodata.readthedocs.io/en/latest/point_data/index.html#how-to) that use some of these different options.

*Note*: Data requested from the `hf_hydrodata` point module is returned in pandas DataFrames. If you want more information on how to work with this data format, please see our [How To: Save and wrangle point observations data](https://hf-hydrodata.readthedocs.io/en/latest/point_data/examples/example_pandas.html).

Let's start by exploring what data is available for HUC8 "02030105" for Water Year 2003. We'll use the parameters:
- `huc_id=["02030105"]`
- `grid="conus2"`
- `date_start="2002-10-01"`
- `date_end="2003-09-30"`

*Note*: Requests can be submitted using two different methods: either as dictionary of options or as named arguments. The following are equivalent. You may see examples using both methods.

In [ ]:
# Option 1: Using dictionary
options = {
    "huc_id": ["02030105"], 
    "grid": "conus2",
    "date_start": "2002-10-01",
    "date_end": "2003-09-30"
}
df = hf.get_site_variables(options)
print(df.shape)

In [ ]:
# Option 2: Using keyword arguments
df = hf.get_site_variables(huc_id=["02030105"], grid="conus2",
                           date_start="2002-10-01", date_end="2003-09-30")
print(df.shape)

In [ ]:
print(f"Number of records: {len(df)}")
df.head(5)

We can see that 544 records were returned. Note that a site could collect data at multiple temporal resolutions (such as daily and hourly). The data availability range for each might be different, which is why some sites have multpile rows per `site_id` for the same `variable`. 

The fields `first_date_data_available` and `last_date_data_available` refer to the earliest and latest dates we have available for the site. The `record_count` field reports on the total number of records over that entire time span and does not relate to the specific `date_start` and `date_end` parameters supplied.

In [ ]:
# Let's define a helper plotting function to use throughout the notebook
def plot_sites_on_map(metadata_df, color_by_site_type=False):
    """
    Plots site locations on an interactive map using Plotly.
    
    Parameters:
    - metadata_df: DataFrame containing site metadata with 'latitude', 'longitude', 'site_name', and 'site_id' columns.
    - color_by_site_type: Boolean indicating whether to color points by site type.
    """
    pio.renderers.default = "notebook"

    center = {
        "lat": metadata_df["latitude"].mean(),
        "lon": metadata_df["longitude"].mean()
    }

    fig = px.scatter_map(
        metadata_df,
        lat="latitude",
        lon="longitude",
        color="site_type" if color_by_site_type else None,
        hover_name="site_name",      # appears on hover
        hover_data=["site_id"],      # additional info
        center=center,
        zoom=8,
        height=600
    )

    fig.update_layout(mapbox_style="carto-voyager")
    return fig

In [ ]:
plot_sites_on_map(df, color_by_site_type=True)

Let's narrow our search down to only daily data from the USGS. To do this we'll use the additional parameters:
- `agency="USGS"`
- `temporal_resolution="daily"`

In [ ]:
df = hf.get_site_variables(huc_id=["02030105"], grid="conus2",
                           date_start="2002-10-01", date_end="2003-09-30",
                           agency="USGS", temporal_resolution="daily")

print(f"Number of records: {len(df)}")
print(f"Number of streamflow sites: {len(df[df['variable'] == 'streamflow'])}")    
print(f"Number of groundwater sites: {len(df[df['variable'] == 'water_table_depth'])}")                      

In [ ]:
# Let's look at the updated map
plot_sites_on_map(df, color_by_site_type=True)

### Example 2: Request USGS streamflow data

In this example, we will use the functions `get_point_data` and `get_point_metadata`. These functions are different from `get_site_variables` in that their purpose is to request information for a specific `dataset`, `variable`, `temporal_resolution`, and `aggregation`. Those parameters are mandatory inputs to these functions. You may additionally include other filters for geographic region, time range, etc. Again the [API Documentation](https://hf-hydrodata.readthedocs.io/en/latest/hf_hydrodata.point.html#module-hf_hydrodata.point) provides detail on all of the available options.

Specific dataset documentation pages such as this [USGS NWIS page](https://hf-hydrodata.readthedocs.io/en/latest/gen_usgs_nwis.html) identify the options available for `variable`, `temporal_resolution`, and `aggregation` within each `dataset`. 

Let's request daily USGS streamflow data and site metadata for the same HUC8 "02030105" for Water Year 2003.
- `dataset="usgs_nwis"`
- `variable="streamflow"`
- `temporal_resolution="daily"`
- `aggregation="mean"`
- `huc_id=["02030105"]`
- `grid="conus2"`
- `date_start="2002-10-01"`
- `date_end="2003-09-30"`

In [ ]:
# We'll use the dictionary approach to specify our options, since we'll use the same options for both data and metadata requests.
options = {
    "dataset": "usgs_nwis",
    "variable": "streamflow",
    "temporal_resolution": "daily",
    "aggregation": "mean",
    "huc_id": ["02030105"], 
    "grid": "conus2",
    "date_start": "2002-10-01",
    "date_end": "2003-09-30"
}

In [ ]:
# Let's request site-level metadata first. 
streamflow_metadata_df = hf.get_point_metadata(options)

print(f"Number of streamflow sites: {len(streamflow_metadata_df)}")
streamflow_metadata_df.head(5)

In [ ]:
plot_sites_on_map(streamflow_metadata_df, color_by_site_type=False)

In [ ]:
# Now let's request data.
streamflow_data_df = hf.get_point_data(options)

streamflow_data_df.head(5)

In [ ]:
# Set the index to data type 'datetime'
streamflow_data_df.set_index("date", inplace=True)
streamflow_data_df.index = pd.to_datetime(streamflow_data_df.index)

# Set units
units = "m3/s"

# Plot
plt.plot(streamflow_data_df)
plt.xlabel("Date")
plt.ylabel(f"Streamflow ({units})")
plt.title(f"Streamflow for for Water Year 2003 in HUC 02030105")

# Create a legend of corresponding site names
# (remove or adjust the bbox_to_anchor parameter to move legend around)
site_names = streamflow_metadata_df['site_name']
plt.legend(site_names, bbox_to_anchor=(1.05, 1))

plt.show()

*Explore*: How would you request `water_table_depth` data from the USGS NWIS for the same region and time period?

### Example 3: Request SNOTEL Snow Water Equivalent data

Here we will change the request to be for `dataset="snotel"` and `variable="swe"`. Because SNOTEL reports there data as start-of-day daily aggregations, we use `temporal_resolution="daily"` and `aggregation="sod"`.

Our previous HUC in New Jersey does not have any SNOTEL stations, so we'll choose a different HUC located in Colorado.
- `dataset="snotel"`
- `variable="swe"`
- `temporal_resolution="daily"`
- `aggregation="sod"`
- `huc_id=["14050001"]`
- `grid="conus2"`
- `date_start="2002-10-01"`
- `date_end="2003-09-30"`

In [ ]:
options = {
    "dataset": "snotel",
    "variable": "swe",
    "temporal_resolution": "daily",
    "aggregation": "sod",
    "huc_id": ["14050001"], 
    "grid": "conus2",
    "date_start": "2002-10-01",
    "date_end": "2003-09-30"
}

In [ ]:
# Let's request site-level metadata first. 
swe_metadata_df = hf.get_point_metadata(options)

print(f"Number of SWE sites: {len(swe_metadata_df)}")
swe_metadata_df.head(5)

In [ ]:
plot_sites_on_map(swe_metadata_df, color_by_site_type=False)

In [ ]:
# Now let's request data.
swe_data_df = hf.get_point_data(options)

swe_data_df.head(5)

In [ ]:
# Set the index to data type 'datetime'
swe_data_df.set_index("date", inplace=True)
swe_data_df.index = pd.to_datetime(swe_data_df.index)

# Set units
units = "mm"

# Plot
plt.plot(swe_data_df)
plt.xlabel("Date")
plt.ylabel(f"SWE ({units})")
plt.title(f"SWE for Water Year 2003 in HUC 14050001")

# Create a legend of corresponding site names
# (remove or adjust the bbox_to_anchor parameter to move legend around)
site_names = swe_metadata_df['site_name']
plt.legend(site_names, bbox_to_anchor=(1.05, 1))

plt.show()

### Wrapping Up: Citing Sources

Each `dataset` has different citation requirements. Use the `get_citations` function within `hf_hydrodata` to see how best to cite the data used.

In [ ]:
hf.get_citations(dataset="usgs_nwis")

In [ ]:
hf.get_citations(dataset="snotel")